## What is Multiple Document RAG?   

Multiple Document RAG is an extension of Retrieval-Augmented Generation (RAG) where the AI retrieves information from multiple documents instead of a single file.   

Workflow:   

Multiple PDFs   
      ↓   
Load Documents   
      ↓   
Split into Chunks   
      ↓   
Generate Embeddings  
      ↓  
Store in FAISS   
      ↓    
Retriever   
      ↓   
    Gemini   
      ↓   
    Answer

In [1]:
from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

from google import genai

import os

C:\Users\Dell\AppData\Local\Temp\ipykernel_6344\2954837846.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


**Load Multiple PDFs**

In [2]:
folder = "day16_pdfs"

documents = []

for file in os.listdir(folder):

    if file.endswith(".pdf"):

        loader = PyPDFLoader(os.path.join(folder, file))

        documents.extend(loader.load())

print("Pages Loaded:", len(documents))

Pages Loaded: 121


**Check Documents**

In [3]:
print(documents[0].metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'day16_pdfs\\attension-is-all-u-need.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}


**verify source and page together**

In [4]:
for i in range(20):
    print(i,
          documents[i].metadata["source"],
          "Page:", documents[i].metadata["page"])

0 day16_pdfs\attension-is-all-u-need.pdf Page: 0
1 day16_pdfs\attension-is-all-u-need.pdf Page: 1
2 day16_pdfs\attension-is-all-u-need.pdf Page: 2
3 day16_pdfs\attension-is-all-u-need.pdf Page: 3
4 day16_pdfs\attension-is-all-u-need.pdf Page: 4
5 day16_pdfs\attension-is-all-u-need.pdf Page: 5
6 day16_pdfs\attension-is-all-u-need.pdf Page: 6
7 day16_pdfs\attension-is-all-u-need.pdf Page: 7
8 day16_pdfs\attension-is-all-u-need.pdf Page: 8
9 day16_pdfs\attension-is-all-u-need.pdf Page: 9
10 day16_pdfs\attension-is-all-u-need.pdf Page: 10
11 day16_pdfs\attension-is-all-u-need.pdf Page: 11
12 day16_pdfs\attension-is-all-u-need.pdf Page: 12
13 day16_pdfs\attension-is-all-u-need.pdf Page: 13
14 day16_pdfs\attension-is-all-u-need.pdf Page: 14
15 day16_pdfs\bert.pdf Page: 0
16 day16_pdfs\bert.pdf Page: 1
17 day16_pdfs\bert.pdf Page: 2
18 day16_pdfs\bert.pdf Page: 3
19 day16_pdfs\bert.pdf Page: 4


In [5]:
for i in range(121):
    print(i,
          documents[i].metadata["source"],
          "Page:", documents[i].metadata["page"])

0 day16_pdfs\attension-is-all-u-need.pdf Page: 0
1 day16_pdfs\attension-is-all-u-need.pdf Page: 1
2 day16_pdfs\attension-is-all-u-need.pdf Page: 2
3 day16_pdfs\attension-is-all-u-need.pdf Page: 3
4 day16_pdfs\attension-is-all-u-need.pdf Page: 4
5 day16_pdfs\attension-is-all-u-need.pdf Page: 5
6 day16_pdfs\attension-is-all-u-need.pdf Page: 6
7 day16_pdfs\attension-is-all-u-need.pdf Page: 7
8 day16_pdfs\attension-is-all-u-need.pdf Page: 8
9 day16_pdfs\attension-is-all-u-need.pdf Page: 9
10 day16_pdfs\attension-is-all-u-need.pdf Page: 10
11 day16_pdfs\attension-is-all-u-need.pdf Page: 11
12 day16_pdfs\attension-is-all-u-need.pdf Page: 12
13 day16_pdfs\attension-is-all-u-need.pdf Page: 13
14 day16_pdfs\attension-is-all-u-need.pdf Page: 14
15 day16_pdfs\bert.pdf Page: 0
16 day16_pdfs\bert.pdf Page: 1
17 day16_pdfs\bert.pdf Page: 2
18 day16_pdfs\bert.pdf Page: 3
19 day16_pdfs\bert.pdf Page: 4
20 day16_pdfs\bert.pdf Page: 5
21 day16_pdfs\bert.pdf Page: 6
22 day16_pdfs\bert.pdf Page: 7
23 day1

**exact location of each file**

In [6]:
for i, doc in enumerate(documents):
    if doc.metadata["page"] == 0:
        print(i, "->", doc.metadata["source"])

0 -> day16_pdfs\attension-is-all-u-need.pdf
15 -> day16_pdfs\bert.pdf
31 -> day16_pdfs\gemini.pdf


**Split Documents**

In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 863


**Embeddings**

In [8]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

**Build FAISS**

In [9]:
vector_db = FAISS.from_documents(
    chunks,
    embeddings
)

print("Vector Database Ready!")

Vector Database Ready!


**Retriever**

In [10]:
retriever = vector_db.as_retriever(
    search_kwargs={"k":3}
)

**Query**

In [11]:
question = "What is Transformer?"

**Retrieve**

In [12]:
docs = retriever.invoke(question)

for doc in docs:

    print("="*60)

    print(doc.metadata)

    print()

    print(doc.page_content)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'day16_pdfs\\attension-is-all-u-need.pdf', 'total_pages': 15, 'page': 2, 'page_label': '3'}

Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creation

**As we can see the retrieved chunks may come from different PDFs**

**Create Context**

In [13]:
context = "\n\n".join(
    [doc.page_content for doc in docs]
)

**Prompt**

In [14]:
prompt = f"""
You are an AI assistant.

Answer ONLY using the retrieved context.

If the answer is not available, reply:

"I couldn't find that information."

Context:

{context}

Question:

{question}

Answer:
"""

**Gemini**

In [15]:
client = genai.Client(api_key="YOUR_API_KEY")
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

The Transformer is a model architecture that uses stacked self-attention and point-wise, fully connected layers for both the encoder and decoder.
